El notebook está en `notebooks/` y el código del proyecto en `src/`. Para poder importarlo se agrega la carpeta raíz del proyecto a la lista de lugares donde Python busca módulos. `sys.path` solo acepta texto, por eso se convierte con `str()`, y se agrega únicamente si todavía no está.

In [ ]:
import sys
from pathlib import Path

raiz = str(Path.cwd().parent)
if raiz not in sys.path:
    sys.path.append(raiz)

from src import config, data_loader, tecnologia

print("Raíz del proyecto:", raiz)
print("Carpeta de datos: ", config.DIR_DATOS)

Se buscan los archivos de la categoría de tecnología y se agrupan por año (el año sale del nombre del archivo, por ejemplo `EPH_usu_hog_tic_t424.txt` es hogares, 2024). Un año solo se puede analizar si tiene **los dos** archivos.

`ANIOS = None` analiza todos los años disponibles. Para analizar solo algunos, se escribe una lista, por ejemplo `["2023", "2024"]`.

In [ ]:
CATEGORIA = config.CAT_CIENCIA_TECNOLOGIA
ANIOS = None

archivos = data_loader.buscar_archivos(categoria=CATEGORIA, anios=ANIOS)
pares = tecnologia.pares_por_anio(archivos)
anios_detectados = sorted(pares)

print("Años detectados:", anios_detectados)
for anio in anios_detectados:
    for tipo, ruta in pares[anio].items():
        print(f"  {anio}  {tipo:11}", ruta.name if ruta else "FALTA")

completos = [a for a in anios_detectados if all(pares[a].values())]
racha = tecnologia.anios_consecutivos(completos)
print(f"\nAños con hogares e individuos: {completos}")
if racha < config.MIN_ANIOS_TIC:
    print(
        f"ATENCIÓN: hay {racha} año(s) consecutivo(s); el enunciado pide "
        f"al menos {config.MIN_ANIOS_TIC}. Copiar los archivos de los años "
        "que faltan en la carpeta de datos y volver a ejecutar."
    )
else:
    print(
        f"Hay {racha} años consecutivos: se cumple el mínimo de "
        f"{config.MIN_ANIOS_TIC}."
    )

Se compara, entre todos los años disponibles, el **conjunto de nombres de columnas** de los archivos de hogares y, por separado, el de los archivos de individuos. Se compara cada archivo contra el primero, sin importar el orden de las columnas, y se informa qué columnas faltan o sobran. Funciona igual con 3 años o con todos los que haya en la carpeta.

Cuidado con interpretar el resultado: con un solo año no hay nada que comparar, por lo que la comparación no prueba consistencia.

In [ ]:
for tipo in (config.TIPO_HOGAR, config.TIPO_INDIVIDUO):
    rutas = [pares[anio][tipo] for anio in completos]
    resultado = tecnologia.comparar_columnas(rutas)
    print(
        f"Archivos de {tipo}: {resultado['archivos_comparados']} comparados "
        f"(referencia: {resultado['referencia']})"
    )
    suficientes = resultado["archivos_comparados"] >= config.MIN_ANIOS_TIC
    if not suficientes:
        print("  Pocos períodos: la comparación no es concluyente.")
    if resultado["consistente"] and suficientes:
        print("  Las columnas son consistentes entre períodos.")
    elif resultado["consistente"]:
        print("  Sin diferencias entre los períodos disponibles.")
    for diferencia in resultado["diferencias"]:
        print("  DIFERENCIA en", diferencia["archivo"], "->", diferencia)

Cada hogar se identifica con la tupla `(CODUSU, NRO_HOGAR, AGLOMERADO)`. Para cada año se verifica que:

- no haya **hogares repetidos** (la tupla debe ser única en el archivo de hogares);
- todo **individuo** pertenezca a un hogar que exista en el archivo de hogares;
- todo **hogar** tenga al menos un individuo.

Si un archivo no se puede leer, se informa el error y se sigue con el año siguiente.

In [ ]:

integridad = {}
for anio in completos:
    try:
        filas_h = data_loader.leer_filas(pares[anio][config.TIPO_HOGAR])
        filas_i = data_loader.leer_filas(pares[anio][config.TIPO_INDIVIDUO])
    except (OSError, ValueError) as error:
        print(f"{anio}: no se pudo leer ({error})")
        continue
    resultado = tecnologia.validar_integridad(filas_h, filas_i)
    integridad[anio] = resultado
    print(
        f"{anio}: {resultado['total_hogares']} hogares "
        f"({resultado['hogares_unicos']} únicos), "
        f"{resultado['total_individuos']} individuos | "
        f"individuos sin hogar: {resultado['individuos_sin_hogar']} | "
        f"hogares sin individuos: {resultado['hogares_sin_individuos']} | "
        f"integridad correcta: {resultado['integridad_correcta']}"
    )

| Indicador | De dónde sale |
|---|---|
| **Computadora** | `IH_II_01` del archivo de **hogares** (1 = sí, 2 = no) |
| **Internet** | `IH_II_02` del archivo de **hogares** (1 = sí, 2 = no) |
| **Celular** | No está en hogares: sale de **individuos** (`IP_III_06`, la persona usó celular). Un hogar tiene celular si **al menos una persona** del hogar lo usó, y la unión se hace con la tupla de la sección 4 |

Decisiones de cálculo (también quedan escritas en el informe JSON):

- **Ponderación:** `POND_TIC`, el factor de expansión del módulo de tecnología. Sin ponderar se describe solo la muestra.
- **Región:** el archivo de tecnología no trae `REGION`; se obtiene del aglomerado (`config.REGION_DE_AGLOMERADO`).
- **Hogares sin dato** (código 9 u otro valor distinto de 1 y 2): se excluyen del denominador de ese indicador y se cuentan aparte.
- Se informan los **nombres** de región y de aglomerado, no solo los códigos.

In [ ]:
resultados = {}
for anio in completos:
    try:
        filas_h = data_loader.leer_filas(pares[anio][config.TIPO_HOGAR])
        filas_i = data_loader.leer_filas(pares[anio][config.TIPO_INDIVIDUO])
    except (OSError, ValueError) as error:
        print(f"{anio}: no se pudo leer ({error})")
        continue
    celular = tecnologia.celular_por_hogar(filas_i)
    resultado = tecnologia.calcular_penetracion(filas_h, celular)
    resultado["integridad"] = integridad.get(anio)
    resultados[anio] = resultado
    print(
        f"{anio}: {resultado['total']['hogares_muestra']} hogares "
        f"analizados | sin peso válido: {resultado['sin_peso']} | "
        f"sin región: {resultado['sin_region']}"
    )

Porcentaje de hogares con computadora, internet y celular, ponderado con `POND_TIC`.

In [ ]:
def mostrar_tabla(titulo, bloques):
    """Imprime los porcentajes de cada grupo (región o aglomerado).

    Args:
        titulo: título de la tabla.
        bloques: lista de bloques de calcular_penetracion (nombre, hogares y
            porcentajes de computadora, internet y celular).
    """
    print(titulo)
    print(
        f"{'Grupo':34}{'Hogares':>12}{'Computadora':>13}"
        f"{'Internet':>10}{'Celular':>9}"
    )
    for b in bloques:
        hogares = f"{b['hogares_ponderados']:,}".replace(",", ".")
        print(
            f"{b['nombre']:34}{hogares:>12}{b['pct_computadora']:>13}"
            f"{b['pct_internet']:>10}{b['pct_celular']:>9}"
        )
    print()


for anio, resultado in resultados.items():
    print("=" * 78)
    print("AÑO", anio)
    print("=" * 78)
    mostrar_tabla(
        "Total y regiones (% de hogares)",
        [resultado["total"]] + list(resultado["por_region"].values()),
    )

In [ ]:
for anio, resultado in resultados.items():
    mostrar_tabla(
        f"Aglomerados, año {anio} (% de hogares)",
        list(resultado["por_aglomerado"].values()),
    )

### Efecto de ponderar

El enunciado pide usar el factor de expansión apropiado. Para ver por qué importa, se compara el porcentaje del total con y sin ponderar: sin ponderar solo se describe la muestra, y los hogares de distintos aglomerados no representan la misma cantidad de hogares reales.

In [ ]:
for anio, resultado in resultados.items():
    total = resultado["total"]
    print(f"Año {anio} (total de aglomerados)")
    for indicador in tecnologia.INDICADORES:
        ponderado = total["pct_" + indicador]
        muestra = total["pct_" + indicador + "_sin_ponderar"]
        print(
            f"  {indicador:12} ponderado: {ponderado:6}%   "
            f"sin ponderar: {muestra:6}%"
        )

Todo el análisis se exporta a `output/tecnologia_informe.json`, con las definiciones de cómo se calculó (`notas`) y los resultados de cada año por región y por aglomerado.

**Si el archivo ya existe**, el programa no lo pisa en silencio. Hay dos opciones y se elige en `config.MODO_ARCHIVO_EXISTENTE`:

- **`"preguntar"` (Opción A):** pregunta si se quiere sobrescribir. Si no hay teclado disponible (por ejemplo, al ejecutar el notebook de forma automática) no sobrescribe y usa la otra opción.
- **`"timestamp"` (Opción B):** guarda un archivo nuevo con la fecha y la hora en el nombre (por ejemplo `tecnologia_informe_20261104_153000.json`).

La existencia del archivo se verifica con `Path.exists()`, dentro de `data_loader.guardar_json`.

In [ ]:
metadatos = {
    "categoria": CATEGORIA,
    "anios": sorted(resultados),
    "anios_consecutivos": racha,
    "ponderador": config.COL_POND_TIC,
}
informe = tecnologia.armar_informe_tecnologia(resultados, metadatos)

ruta_informe = config.DIR_SALIDA / config.ARCHIVO_TECNOLOGIA
guardado = data_loader.guardar_json(informe, ruta_informe)
print("Informe guardado en:", guardado)

## Conclusiones

*(Valores de 2024; actualizar al agregar años.)*

- Los archivos de hogares e individuos de 2024 son **consistentes entre sí**: 16.484 hogares con tupla única, 45.192 individuos, ningún individuo sin hogar y ningún hogar sin individuos.
- Alrededor de **60% de los hogares tiene computadora**, **94% acceso a internet** y **97% celular** (ponderado). La computadora es la que más varía entre regiones: del 51% en el Noreste al 67% en la Patagonia.
- Ponderar importa: sin ponderar, la computadora daría 56,6% en lugar de 60,3%.
- El celular se resuelve a nivel hogar con "al menos una persona lo usó": 16.031 hogares sí, 452 no y 1 sin dato.